In [ ]:
!pip install pyspark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col
import time

# --------------------------------
# Create Spark Session
# --------------------------------
spark = SparkSession.builder \
    .appName("NewsPulse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("News Pulse Streaming Started...")

# --------------------------------
# Simulated Streaming Loop
# --------------------------------
for batch in range(5):

    # Read CSV dataset
    news_df = spark.read \
        .option("header", "true") \
        .csv("news_dataset.csv")

    # --------------------------------
    # Processing 1: Headlines by Source
    # --------------------------------
    source_counts = news_df.groupBy("source").count()

    print("\n===== Headlines By Source =====")
    source_counts.show(truncate=False)

    # --------------------------------
    # Processing 2: Trending Keywords
    # --------------------------------
    words_df = news_df.select(
        explode(
            split(col("headline"), " ")
        ).alias("word")
    )

    word_counts = words_df.groupBy("word").count()

    print("\n===== Trending Keywords =====")
    word_counts.show(truncate=False)

    # --------------------------------
    # Processing 3: Total Headlines
    # --------------------------------
    total_headlines = news_df.count()

    print(f"\nTotal Headlines Processed: {total_headlines}")

    # --------------------------------
    # Simulate Streaming Delay
    # --------------------------------
    print("\nWaiting for next streaming batch...\n")

    time.sleep(5)

print("Streaming Finished.")